In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [12]:
data=pd.read_csv('C:\\Users\\adaml\\Documents\\AI_ML_DL\\Projects\\1 - Telecom Customer Churn Prediction\\raw_data\\WA_Fn-UseC_-Telco-Customer-Churn.csv')
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
# Check for NaN and whitespace-only values, then clean them
blank_count = data.replace(r'^\s*$', np.nan, regex=True).isna().sum().sum()
nan_count = data.isna().sum().sum()
print(f'NaN values before cleaning: {nan_count}')
print(f'Whitespace-only values before cleaning: {blank_count - nan_count}')

data = data.replace(r'^\s*$', np.nan, regex=True)

for column in data.columns:
    if data[column].isna().any():
        if pd.api.types.is_numeric_dtype(data[column]):
            data[column] = data[column].fillna(data[column].median())
        else:
            data[column] = data[column].fillna(data[column].mode(dropna=True)[0])

print(f'NaN values after cleaning: {data.isna().sum().sum()}')

NaN values before cleaning: 0
Whitespace-only values before cleaning: 11
NaN values after cleaning: 0


In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [15]:
data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [16]:
data[["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]] = data[["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]].replace({"Yes": 1, "No": 0})

C:\Users\adaml\AppData\Local\Temp\ipykernel_38744\2474136029.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]] = data[["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]].replace({"Yes": 1, "No": 0})


In [17]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.5,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [18]:
cols = ["PaymentMethod", "Contract", "StreamingMovies", "StreamingTV","TechSupport","DeviceProtection","OnlineBackup","OnlineSecurity","InternetService","MultipleLines"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(data[cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cols),
    index=data.index
)

data = data.drop(columns=cols).join(encoded_df)

In [19]:
data["gender"].replace({"Male": 1, "Female": 0}, inplace=True)

C:\Users\adaml\AppData\Local\Temp\ipykernel_38744\1037728577.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["gender"].replace({"Male": 1, "Female": 0}, inplace=True)
C:\Users\adaml\AppData\Local\Temp\ipykernel_38744\1037728577.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["gende

In [20]:
mapping = data["customerID"]
mapping.head()

0    7590-VHVEG
1    5575-GNVDE
2    3668-QPYBK
3    7795-CFOCW
4    9237-HQITU
Name: customerID, dtype: object

In [21]:
data.drop(columns=["customerID"], inplace=True)

In [22]:
data.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'StreamingTV_No',
       'StreamingTV_No internet service', 'StreamingTV_Yes', 'TechSupport_No',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'DeviceProtection_No', 'DeviceProtection_No internet service',
       'DeviceProtection_Yes', 'OnlineBackup_No',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'OnlineSecurity_No', 'OnlineSecurity_No internet service',
       'OnlineSecurity_Yes', 'InternetService_DSL',
       'InternetService_Fiber optic', 'Inte

In [23]:
data.dtypes

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                                object
Churn                                        int64
PaymentMethod_Bank transfer (automatic)    float64
PaymentMethod_Credit card (automatic)      float64
PaymentMethod_Electronic check             float64
PaymentMethod_Mailed check                 float64
Contract_Month-to-month                    float64
Contract_One year                          float64
Contract_Two year                          float64
StreamingMovies_No                         float64
StreamingMovies_No internet service        float64
StreamingMovies_Yes            

In [24]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

In [25]:
data.dtypes

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn                                        int64
PaymentMethod_Bank transfer (automatic)    float64
PaymentMethod_Credit card (automatic)      float64
PaymentMethod_Electronic check             float64
PaymentMethod_Mailed check                 float64
Contract_Month-to-month                    float64
Contract_One year                          float64
Contract_Two year                          float64
StreamingMovies_No                         float64
StreamingMovies_No internet service        float64
StreamingMovies_Yes            

In [26]:
x = data.drop(columns=["Churn"])
y = data["Churn"]

In [27]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
x_train.to_csv('C:\\Users\\adaml\\Documents\\AI_ML_DL\\Projects\\1 - Telecom Customer Churn Prediction\\ready_data\\x_train.csv', index=False)
x_test.to_csv('C:\\Users\\adaml\\Documents\\AI_ML_DL\\Projects\\1 - Telecom Customer Churn Prediction\\ready_data\\x_test.csv', index=False)
y_train.to_csv('C:\\Users\\adaml\\Documents\\AI_ML_DL\\Projects\\1 - Telecom Customer Churn Prediction\\ready_data\\y_train.csv', index=False)
y_test.to_csv('C:\\Users\\adaml\\Documents\\AI_ML_DL\\Projects\\1 - Telecom Customer Churn Prediction\\ready_data\\y_test.csv', index=False)